# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Question (Lane 4):** among visible pages, which ones should an editor review first because their CTR gap vs. position is *real and lasting*, not a one-month wobble?

**Setup (starter slice, same data as my Week-4 baseline):**
- **Population:** pages at positions 1–20 with ≥ 1,000 impressions in 90 days (the Week-4 population), plus ≥ 100 impressions in each 30-day window so both CTRs are measurable.
- **Decision moment:** the start of the last 30 days. **Features** come from the *previous* 30 days plus static page context. **Label** `stays_under` = CTR in the **last** 30 days is below half of the pooled CTR of the page's narrow position bucket. This is an observed later-window outcome.
- **Baseline (frozen from Week 4):** the same rule, *LOW_CTR_FOR_POSITION scored by missing clicks*, re-scored on the feature window. The Week-4 queue itself used 90-day totals, which contain the label window, so scoring it against this label would be leakage.

**Methods, simplest first:**
1. **Logistic regression.** Readable; shows whether a linear blend of signals already beats the rule.
2. **Random forest.** Handles thresholds and interactions (e.g. "low CTR *and* high volume *and* position 1–3") without hand-tuning.
3. **Gradient boosting** (shallow, depth 3) as a check on the forest.
4. **Forest × potential.** The forest gives *probability the gap is real*; the baseline knows *how many clicks are at stake*. Multiplying them ranks by expected missed clicks.

A probability alone is the wrong ranking objective for an editor with limited hours, which is why method 4 exists. The table in section 3 shows why.

In [1]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
warnings.filterwarnings("ignore")
SEED = 42
print("scikit-learn", sklearn.__version__, "| seed", SEED)

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw/content_refresh_anonymized.csv").exists())
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")

d = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 1000)
       & (df["impressions_prev_30d"] >= 100) & (df["impressions_last_30d"] >= 100)].copy()
d["bucket"] = pd.cut(d["avg_position"], [0, 3, 5, 10, 15, 20], labels=["1-3", "3-5", "5-10", "10-15", "15-20"])
for w in ["prev", "last"]:
    d[f"ctr_{w}"] = 100 * d[f"clicks_{w}_30d"] / d[f"impressions_{w}_30d"]
    t = d.groupby("bucket", observed=True)[[f"clicks_{w}_30d", f"impressions_{w}_30d"]].sum()
    d[f"exp_{w}"] = d["bucket"].map(100 * t[f"clicks_{w}_30d"] / t[f"impressions_{w}_30d"]).astype(float)

# LABEL: observed in the later window only
d["y"] = (d["ctr_last"] < 0.5 * d["exp_last"]).astype(int)

# Frozen Week-4 rule, scored on the FEATURE window
d["potential"] = (d["exp_prev"] / 100 * d["impressions_prev_30d"] - d["clicks_prev_30d"]).clip(lower=0)
d["baseline_score"] = np.where(d["ctr_prev"] < 0.5 * d["exp_prev"], d["potential"], 0.0)

# Features: feature window + static context only
d["ratio_prev"] = d["ctr_prev"] / d["exp_prev"]
d["log_imp_prev"] = np.log1p(d["impressions_prev_30d"])
d["log_clk_prev"] = np.log1p(d["clicks_prev_30d"])
d["log_sess_prev"] = np.log1p(d["sessions_prev_30d"])
d["has_word_count"] = d["word_count"].notna().astype(int)
d["word_count_f"] = d["word_count"].fillna(0)
d = pd.get_dummies(d, columns=["main_intent", "content_type"], dummy_na=True, dtype=int)
CAT = [c for c in d.columns if c.startswith(("main_intent_", "content_type_"))]
FEATURES = ["log_imp_prev", "log_clk_prev", "ctr_prev", "ratio_prev", "avg_position",
            "log_sess_prev", "content_age_days", "has_word_count", "word_count_f"] + CAT
d = d.reset_index(drop=True)
print(f"Modeling frame: {len(d):,} pages, {d['client_id'].nunique()} clients, label base rate {d['y'].mean():.1%}")
print("Features:", ", ".join(FEATURES))

scikit-learn 1.9.1 | seed 42
Modeling frame: 9,664 pages, 28 clients, label base rate 37.8%
Features: log_imp_prev, log_clk_prev, ctr_prev, ratio_prev, avg_position, log_sess_prev, content_age_days, has_word_count, word_count_f, main_intent_commercial, main_intent_informational, main_intent_navigational, main_intent_transactional, main_intent_nan, content_type_comparison article, content_type_feedly article, content_type_keyword article, content_type_nan


## 2. Split design

**5-fold `GroupKFold` on `client_id`.** Every test fold contains only clients the model never saw in training, and all methods, the baseline included, are scored on **the same five folds**.

Why this is the honest split here:
- Pages of one client share templates, brand strength and SERP context. A random split would let the model memorize a client's "style" and look better than it would on a new client.
- FlyRank onboards new clients. "Works on unseen clients" is the claim that matters.
- I report the **mean and the per-fold spread**. With only 28 clients, one unusual client can move a fold a lot, so a single number would overstate certainty.

**Deliberately excluded features** (would leak the label window or the label itself): `trend_pct`, `trend_direction`, every `*_last_30d` column, and 90-day totals of clicks/impressions/sessions/engagement. `days_since_last_update` is excluded too, because an update during the label window would show up in it. `client_id` / `content_id` are used only for grouping.

**Known overlap I can't remove in the starter slice:** `avg_position` is a 90-day average, so it overlaps the label window. The baseline and the models all share it, so the comparison stays fair, but absolute scores may be slightly optimistic. The warehouse version would rebuild position from the feature window only.

In [2]:
FORBIDDEN = {"trend_pct", "trend_direction", "is_declining_label", "impressions_last_30d", "clicks_last_30d",
             "sessions_last_30d", "clicks_90d", "impressions_90d", "sessions_90d", "engagement_rate",
             "days_since_last_update", "client_id", "content_id"}
assert FORBIDDEN.isdisjoint(FEATURES), "a forbidden column slipped into FEATURES"

folds = list(GroupKFold(n_splits=5).split(d, d["y"], d["client_id"]))
for i, (tr, te) in enumerate(folds, 1):
    shared = set(d.loc[tr, "client_id"]) & set(d.loc[te, "client_id"])
    print(f"fold {i}: train {len(tr):,} pages / test {len(te):,} pages from {d.loc[te, 'client_id'].nunique()} clients | "
          f"clients in both: {len(shared)} | test base rate {d.loc[te, 'y'].mean():.1%}")

fold 1: train 5,879 pages / test 3,785 pages from 1 clients | clients in both: 0 | test base rate 45.2%
fold 2: train 8,181 pages / test 1,483 pages from 4 clients | clients in both: 0 | test base rate 38.8%
fold 3: train 8,194 pages / test 1,470 pages from 4 clients | clients in both: 0 | test base rate 23.5%
fold 4: train 8,201 pages / test 1,463 pages from 9 clients | clients in both: 0 | test base rate 40.6%
fold 5: train 8,201 pages / test 1,463 pages from 10 clients | clients in both: 0 | test base rate 29.5%


## 3. Train + compare vs my baseline

Metrics, all computed **inside each held-out fold** and then averaged:
- **ROC-AUC:** overall ranking quality.
- **Precision@20 / @50:** of the top 20 / 50, how many really stayed under. This is the editor's view.
- **Clicks-gap captured@50:** of all clicks actually missed in the label window (by pages that stayed under), what share sits in the top 50. This is the value view, and a queue of tiny pages can't fake it.
- **Base rate:** what random picking would get.

In [3]:
def p_at_k(y, s, k):
    return y[np.argsort(-s, kind="mergesort")[:k]].mean()

def gap_at_k(gap, s, k):
    return gap[np.argsort(-s, kind="mergesort")[:k]].sum() / gap.sum()

d["gap_last"] = np.where(d["y"] == 1, (d["exp_last"] / 100 * d["impressions_last_30d"] - d["clicks_last_30d"]).clip(lower=0), 0.0)

MODELS = {
    "Logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Random forest": lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=SEED, n_jobs=-1),
    "Gradient boosting (depth 3)": lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, random_state=SEED),
}
rows, oof_rf = [], np.zeros(len(d))
for fold, (tr, te) in enumerate(folds, 1):
    y, gap = d["y"].values[te], d["gap_last"].values[te]
    scores = {"Baseline rule (Week 4, frozen)": d["baseline_score"].values[te]}
    for name, make in MODELS.items():
        m = make().fit(d.loc[tr, FEATURES], d.loc[tr, "y"])
        scores[name] = m.predict_proba(d.loc[te, FEATURES])[:, 1]
    oof_rf[te] = scores["Random forest"]
    scores["Random forest x potential"] = scores["Random forest"] * d["potential"].values[te]
    for name, s in scores.items():
        rows.append({"method": name, "fold": fold, "roc_auc": roc_auc_score(y, s), "p@20": p_at_k(y, s, 20),
                     "p@50": p_at_k(y, s, 50), "clicks_gap@50": gap_at_k(gap, s, 50), "base_rate": y.mean()})

res = pd.DataFrame(rows)
order = ["Baseline rule (Week 4, frozen)", "Logistic regression", "Random forest", "Gradient boosting (depth 3)", "Random forest x potential"]
table = res.groupby("method")[["roc_auc", "p@20", "p@50", "clicks_gap@50", "base_rate"]].mean().reindex(order)
table["p@50 fold range"] = res.groupby("method")["p@50"].agg(lambda s: f"{s.min():.2f}-{s.max():.2f}").reindex(order)
d["oof_rf"] = oof_rf

out = root / "work/outputs"; out.mkdir(parents=True, exist_ok=True)
(out / "model_metrics.json").write_text(json.dumps({
    "split": "GroupKFold(5) on client_id", "seed": SEED, "sklearn": sklearn.__version__,
    "n_pages": int(len(d)), "n_clients": int(d["client_id"].nunique()), "base_rate": round(float(d["y"].mean()), 4),
    "results": table.drop(columns=["p@50 fold range"]).round(4).reset_index().to_dict(orient="records")}, indent=2))
print("Model vs baseline - same data, same 5 client-grouped folds, same metrics (fold means):")
table.round(3)

Model vs baseline - same data, same 5 client-grouped folds, same metrics (fold means):


,roc_auc,p@20,p@50,clicks_gap@50,base_rate,p@50 fold range
method,,,,,,
"Baseline rule (Week 4, frozen)",0.703,0.76,0.696,0.385,0.355,0.54-0.86
Logistic regression,0.772,0.80,0.768,0.075,0.355,0.54-0.96
Random forest,0.791,0.87,0.804,0.208,0.355,0.62-0.94
Gradient boosting (depth 3),0.779,0.86,0.740,0.273,0.355,0.50-0.96
Random forest x potential,0.717,0.82,0.704,0.406,0.355,0.54-0.84


**What the table says (and doesn't):**
- **On held-out clients, every model measured a higher mean ROC-AUC and precision than the frozen rule.** The random forest had the highest mean precision@50 (about 0.80 vs about 0.70 for the rule), but the per-fold ranges overlap (0.62–0.94 vs 0.54–0.86), so the size of the gain is uncertain.
- **But the forest alone loses on value.** Ranked purely by probability, its top 50 is full of small pages it is sure about. It captures only about a fifth of the missed clicks, against roughly 0.39 for the rule. More complexity didn't automatically mean a better queue.
- **The combined ranking (forest × potential) is the most balanced option observed.** It keeps the rule's value focus (clicks-gap captured@50 about 0.41 vs 0.39 for the rule, within fold noise) at about the same precision. That's a directional result, not a measured improvement.
- **Folds vary a lot** (see the p@50 range). The data is uneven: one client alone is fold 1 (about 39% of pages), while fold 5 has 10 small clients. Base rates per fold run from 24% to 45%. Treat differences of a few points as noise.
- Ranking AUC for *forest × potential* is lower (about 0.72) because the multiplication reorders the whole list by size. It's built for the top of the queue, not for overall separation.

**Directional suggestion for a decision-support queue:** rank by *forest probability × missed-click potential* and keep the rule's reason code (`LOW_CTR_FOR_POSITION`) for explainability. The editor still decides; the score only orders the review.

## 4. Errors and interpretation

What the forest leans on (permutation importance on held-out clients), where it is wrong, and three concrete wrong cases.

In [4]:
# Permutation importance: refit on folds 1-4's training data, measure on fold 5's unseen clients
tr, te = folds[-1]
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=SEED, n_jobs=-1).fit(d.loc[tr, FEATURES], d.loc[tr, "y"])
pi = permutation_importance(rf, d.loc[te, FEATURES], d.loc[te, "y"], scoring="roc_auc", n_repeats=10, random_state=SEED, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False)
print("Permutation importance (drop in held-out ROC-AUC when the feature is shuffled):")
print(imp.head(8).round(4).to_string())

# Where is it wrong? Error rate of the model's top-10% picks, by segment
d["top_pick"] = d.groupby(pd.Series(np.zeros(len(d)), index=d.index))["oof_rf"].rank(pct=True) >= 0.9
picks = d[d["top_pick"]].copy()
picks["false_positive"] = picks["y"] == 0
intent_cols = [c for c in d.columns if c.startswith("main_intent_") and c != "main_intent_nan"]
picks["intent"] = picks[intent_cols].idxmax(axis=1).str.replace("main_intent_", "")
print(f"\nModel's top 10% picks: {len(picks):,} pages, false-positive rate {picks['false_positive'].mean():.1%}")
print("\nFalse-positive rate by position bucket:")
print(picks.groupby("bucket", observed=True)["false_positive"].agg(rate="mean", n="size").round(3).to_string())
print("\nFalse-positive rate by intent:")
print(picks.groupby("intent")["false_positive"].agg(rate="mean", n="size").round(3).to_string())
print("\nFalse positives vs. true positives (medians):")
print(picks.groupby("false_positive")[["impressions_prev_30d", "ctr_prev", "ratio_prev", "avg_position"]].median().round(3).to_string())

Permutation importance (drop in held-out ROC-AUC when the feature is shuffled):
ratio_prev          0.0411
ctr_prev            0.0379
log_clk_prev        0.0284
log_sess_prev       0.0061
log_imp_prev        0.0052
avg_position        0.0050
word_count_f        0.0038
content_age_days    0.0011

Model's top 10% picks: 967 pages, false-positive rate 23.0%

False-positive rate by position bucket:
         rate    n
bucket            
1-3     0.429    7
3-5     0.222   36
5-10    0.160  331
10-15   0.258  310
15-20   0.276  283

False-positive rate by intent:
                rate    n
intent                   
commercial     0.273  161
informational  0.229  663
navigational   0.000    1
transactional  0.183  142

False positives vs. true positives (medians):
                impressions_prev_30d  ctr_prev  ratio_prev  avg_position
false_positive                                                          
False                         1601.0       0.0         0.0         11.10
True           

In [5]:
# Three concrete wrong cases (no IDs): the most confident misses, and the biggest miss in the other direction
fp = d[d["y"] == 0].sort_values("oof_rf", ascending=False).head(2)
fn = d[(d["y"] == 1)].sort_values("gap_last", ascending=False)
fn = fn[fn["oof_rf"] < 0.3].head(1)
cases = pd.concat([fp.assign(case="false positive"), fn.assign(case="false negative")])
cases[["case", "oof_rf", "avg_position", "impressions_prev_30d", "ctr_prev", "exp_prev",
       "impressions_last_30d", "ctr_last", "exp_last"]].round(3).reset_index(drop=True)

,case,oof_rf,avg_position,impressions_prev_30d,ctr_prev,exp_prev,impressions_last_30d,ctr_last,exp_last
0,false positive,0.792,11.9,7705,0.013,0.355,818,1.222,0.416
1,false positive,0.792,9.7,9331,0.021,0.276,1313,0.305,0.302
2,false negative,0.267,8.7,28339,0.191,0.276,86232,0.101,0.302


**Reading the errors:**

- **What it leans on.** The top features are the feature-window CTR signals: CTR vs. its bucket (`ratio_prev`), raw CTR and click volume. That is plausible: in this slice, the strongest measured signal for next month's gap is this month's gap, *measured relative to position*. Nothing looks suspiciously perfect; no single feature carries the model, which is what leakage tends to look like.
- **Where it's wrong.** About 23% of the model's top-10% picks recovered (false positives), vs. a 37.8% base rate. Most picks and most errors sit at positions 5–20; the 1–3 and 3–5 buckets have too few picks to read. False positives have **fewer impressions** than true positives (median ~950 vs ~1,600 in the feature window). Both groups have a median of *zero* clicks that month, so for small pages "zero clicks" is partly luck, and the model can't fully separate a real gap from a thin sample.
- **The three cases.** Both confident false positives had near-zero CTR and then lost most of their impressions in the label window (about 7.7k → 0.8k and 9.3k → 1.3k), while CTR rose to or above expected. That looks like a **query-mix shift**: the broad, low-intent impressions disappeared and the remaining ones clicked. The false negative is the opposite: a large page whose impressions roughly **tripled** (about 28k → 86k) and whose CTR halved. New broad queries diluted CTR, which nothing in the feature window predicted. Both patterns point to the same missing signal: query-level impression mix, which the warehouse has and the starter CSV doesn't.

**Bottom line (careful words):** on held-out clients, a random forest measured a lasting under-capture more precisely than my Week-4 rule. Ranking by *probability × missed-click potential* matched the rule's value at similar precision (differences within fold noise). This is directional, decision-support evidence on a 28-client starter slice, with one known position-window overlap. It is not proof that a title rewrite recovers clicks.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.